# CryptoMind 內容檢查 v7：微調 YuFeng-XGuard-Reason-0.6B（只讀第一個 token）

**開始前，右側 Settings 設定：**
- Accelerator：**GPU T4 x2**（只會用到一張）
- Internet：**On**（要下載 YuFeng 模型，約 1.5 GB）
- Input → **Add Input** → 加入內容檢查資料集（上傳 `cryptomind-moderation-v7.zip` 成同一個資料集的新版本）

**跑法：**
- 想先確認環境：把下面的 `SMOKE` 改成 `True`，跑完幾分鐘沒報錯，再改回 `False`
- 正式訓練：右上 **Save Version → Save & Run All (Commit)**，可以關掉瀏覽器，跑完到 Output 下載 `yufeng-moderation.zip`（約 1.1 GB）


In [ ]:
# 1. 找到上傳的資料、確認有 GPU
import glob
import os

import torch

hits = glob.glob('/kaggle/input/**/train_llm.py', recursive=True)
assert hits, '找不到 train_llm.py：右側 Input → Add Input 加入 v7 版的內容檢查資料集'
DATA = os.path.dirname(hits[0])
print('資料：', DATA, sorted(os.listdir(DATA)))
assert torch.cuda.is_available(), '沒有 GPU：右側 Settings → Accelerator 選 GPU T4 x2'
print('GPU：', torch.cuda.get_device_name(0))


In [ ]:
# 2. 安裝套件（Qwen3 要新版 transformers；LoRA 用 peft）
# Kaggle 預裝的 torchao 0.10 太舊，新版 peft 偵測到就直接報錯（沒裝則略過）；我們用不到，先移除
!pip uninstall -y -q torchao
!pip install -q -U "transformers>=4.56" peft


In [ ]:
# 3. 訓練（正式約 1～3 小時；SMOKE=True 只跑少量確認流程）
SMOKE = False
OUT = '/kaggle/working/yufeng-moderation'
!python {DATA}/train_llm.py --data {DATA} --out {OUT} {'--smoke' if SMOKE else ''}


In [ ]:
# 4. 成績摘要（完整明細在上一格的輸出）
import json

m = json.load(open(f'{OUT}/metrics.json', encoding='utf-8'))
before, after = m['before_finetune'], m['test']
print(f"微調前：不過擋下 {before['block_rate']:.1%}，過被誤擋 {before['false_block_rate']:.1%}")
print(f"微調後：不過擋下 {after['block_rate']:.1%}，過被誤擋 {after['false_block_rate']:.1%}（第 {m['best_epoch']} 輪，門檻 {after['threshold']:.4f}）")
o = after['official_0.5']
print(f"官方規則（最高風險類別 ≥ 0.5）：不過擋下 {o['block_rate']:.1%}，過被誤擋 {o['false_block_rate']:.1%}")
print('達標 ✓' if m['passed'] else '還沒達標 ✗（看上一格「達標線」哪幾條沒過）')


In [ ]:
# 5. 下載：點下面的連結，或右側 Output 找 yufeng-moderation.zip（約 1.1 GB），拿回來給 Claude
from IPython.display import FileLink

os.chdir('/kaggle/working')
FileLink('yufeng-moderation.zip')
